In [1]:
from dotenv import load_dotenv
from pprint import pprint
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

load_dotenv()

True

# Define State Schema

In [2]:
class CustomState(AgentState):
    favourite_colour: str

# Write to state

## Option 1: Model infers state and update it using `@tool` function

In [16]:
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

agent = create_agent(
    model="gpt-5-nano",
    tools=[update_favourite_colour],        # IMPORTANT: Don't forget to add the tool to the agent
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

response = agent.invoke(
    input={ "messages": [HumanMessage(content="My favourite colour is green")]},
    config={"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='0b4c86e7-2787-47ba-9938-139ec2f7374f'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 141, 'total_tokens': 232, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dbg6QXFOg3yNK5ykZgHd6ohTyTQ7S', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df173-271a-7483-bac5-4b21653e5739-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_WqwtvZDt7iFolfMFcTMZdyJP', 'type': 'tool_call'}], invalid_to

## Option 2: Directly passing state value in the input prompt

In [17]:
agent = create_agent(
    model="gpt-5-nano",
    # tools=[update_favourite_colour],        # we can skip this, as already specified state schema and pass the state directly in the input prompt
    checkpointer=InMemorySaver(),
    state_schema=CustomState                    # state schema needs to be specified here
)

response = agent.invoke(
    input={ 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"             # pass state value directly in the input prompt
    },
    config={"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='78e0ecde-0941-488f-ba20-9746f6f33c3e'),
              AIMessage(content="Hello! I'm here and ready to help. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 345, 'prompt_tokens': 12, 'total_tokens': 357, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DbgBOCj1Eo8XpQxBJZDAdwRRmxQAX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df177-dc97-7a11-a86c-bd71a2afcc43-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 345, 'total_tokens':

## Read state

In [24]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


In [28]:
agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [29]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='8d1860de-c02e-4e00-83a0-b650855e055f'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DbgGwYCic6emv5LHQED9oG7RSGqXe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df17d-1d87-7a72-a7bc-6d6fba188ee5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_c8UvZ6navGD6RC3ESt41SZFG', 'type': 'tool_call'}], invalid_

In [30]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='8d1860de-c02e-4e00-83a0-b650855e055f'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DbgGwYCic6emv5LHQED9oG7RSGqXe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df17d-1d87-7a72-a7bc-6d6fba188ee5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_c8UvZ6navGD6RC3ESt41SZFG', 'type': 'tool_call'}], invalid_

# Compare `context`, `state`, and `memory`

## Context

- context = read-only runtime data you provide for this invocation
- context is given to the agent
- Choose context when the data is already known outside the agent and should be treated as runtime dependency/input

## State

- state   = mutable agent data that can change during the run and be checkpointed
- state is owned and updated by the agent
- Choose state when the data is produced, updated, or remembered by the agent during the interaction
- The state schema defines the shape and meaning of that state (**structured conversational memory**)
- Useful for durable, structured facts:
    * Tools can read it directly.
    * You can update it intentionally.
    * You can avoid relying on the model to re-infer it.
    * You can keep it even if old messages are trimmed.
    * You can use it in deterministic program logic.

## Memory

- Combination of `checkpointer` + `config` can make an agent “remember” across calls, used to only persists the agent’s state. 
- **unstructured conversational memory**
- This is useful for conversational continuity, but it doesn't give you the structure and meaning that a state schema provides.
- Potential problems:
    * The value may be buried in a long conversation.
    * The user may contradict themselves later.
    * The model may misread the history.
    * You may trim old messages to save tokens.
    * Tools cannot reliably access the value unless they parse conversation text.